In [1]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("Kaggle Auth Success")

Kaggle Auth Success


In [2]:
api.dataset_download_files(
    'lovishbansal123/sales-of-a-supermarket',
    path='./data',
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/lovishbansal123/sales-of-a-supermarket


In [3]:
import pandas as pd

df = pd.read_csv('./data/supermarket_sales.csv')
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [4]:
# Select relevant columns for customer dimension
dim_customer = df[['Customer type', 'Gender']].drop_duplicates().reset_index(drop=True)

# Create surrogate key (primary key)
dim_customer['customer_id'] = dim_customer.index + 1

# Preview
dim_customer.head()

,Customer type,Gender,customer_id
0,Member,Female,1
1,Normal,Female,2
2,Normal,Male,3
3,Member,Male,4


In [5]:
# Select product-related column
dim_product = df[['Product line']].drop_duplicates().reset_index(drop=True)

# Create surrogate key
dim_product['product_id'] = dim_product.index + 1

# Preview
dim_product.head()

,Product line,product_id
0,Health and beauty,1
1,Electronic accessories,2
2,Home and lifestyle,3
3,Sports and travel,4
4,Food and beverages,5


In [6]:
# Merge customer dimension to get customer_id
df = df.merge(dim_customer, on=['Customer type', 'Gender'])

# Merge product dimension to get product_id
df = df.merge(dim_product, on=['Product line'])

# Create fact table with transactional data
fact_sales = df[['Invoice ID', 'customer_id', 'product_id',
                 'Quantity', 'Unit price', 'Total', 'Date']]

# Preview
fact_sales.head()

,Invoice ID,customer_id,product_id,Quantity,Unit price,Total,Date
0,750-67-8428,1,1,7,74.69,548.9715,1/5/2019
1,226-31-3081,2,2,5,15.28,80.2200,3/8/2019
2,631-41-3108,3,3,7,46.33,340.5255,3/3/2019
3,123-19-1176,4,1,8,58.22,489.0480,1/27/2019
4,373-73-7910,3,4,7,86.31,634.3785,2/8/2019


In [7]:
import sqlite3

# Create/connect to SQLite database file
conn = sqlite3.connect('sales.db')

print("Database created successfully")

Database created successfully


In [9]:
# Load dimension tables
dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)

# Load fact table
fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

print("Data loaded into SQLite successfully")

Data loaded into SQLite successfully


In [10]:
# Check sample data from tables
import pandas as pd

pd.read_sql("SELECT * FROM dim_customer LIMIT 5", conn)

,Customer type,Gender,customer_id
0,Member,Female,1
1,Normal,Female,2
2,Normal,Male,3
3,Member,Male,4


In [12]:
# Rename dimension tables
dim_customer = dim_customer.rename(columns={
    'Customer type': 'customer_type',
    'Gender': 'gender'
})

dim_product = dim_product.rename(columns={
    'Product line': 'product_line'
})

# Rename fact table columns
fact_sales = fact_sales.rename(columns={
    'Invoice ID': 'invoice_id',
    'Quantity': 'quantity',
    'Unit price': 'unit_price',
    'Total': 'total',
    'Date': 'date'
})

In [13]:
dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

1000

In [14]:
pd.read_sql("SELECT * FROM dim_product LIMIT 5", conn)

,product_line,product_id
0,Health and beauty,1
1,Electronic accessories,2
2,Home and lifestyle,3
3,Sports and travel,4
4,Food and beverages,5


In [15]:
# SQL query with joins + aggregation + window function
query = """
SELECT 
    dp.product_line,
    dc.gender,
    
    -- Total sales per group
    SUM(fs.total) AS total_sales,
    
    -- Average sales
    AVG(fs.total) AS avg_sales,
    
    -- Ranking within each product category
    RANK() OVER (
        PARTITION BY dp.product_line 
        ORDER BY SUM(fs.total) DESC
    ) AS rank_within_product

FROM fact_sales fs

-- Join with dimensions
JOIN dim_customer dc ON fs.customer_id = dc.customer_id
JOIN dim_product dp ON fs.product_id = dp.product_id

-- Grouping
GROUP BY dp.product_line, dc.gender;
"""

# Execute query
report = pd.read_sql(query, conn)

# Show result
report

,product_line,gender,total_sales,avg_sales,rank_within_product
0,Electronic accessories,Male,27235.5090,316.691965,1
1,Electronic accessories,Female,27102.0225,322.643125,2
2,Fashion accessories,Female,30437.4000,317.056250,1
3,Fashion accessories,Male,23868.4950,291.079207,2
4,Food and beverages,Female,33170.9175,368.565750,1
5,Food and beverages,Male,22973.9265,273.499125,2
6,Health and beauty,Male,30632.7525,348.099460,1
7,Health and beauty,Female,18560.9865,290.015414,2
8,Home and lifestyle,Female,30036.8775,380.213639,1
9,Home and lifestyle,Male,23825.0355,294.136241,2


In [16]:
#close data base connection
conn.close()